# Stage-2 Gate 0.1 legacy-inventory recovery

Recovery-only, unexecuted operator for the completed v7 evidence. **Runtime → Run all** requires only Google Drive authorization. A CPU High-RAM runtime is recommended; the GPU is unused. It performs early dependency, disk-capacity, topology, and metadata preflights, reuses and verifies the sealed A100/20/200-step evidence on CPU, imports P:0006 for development/model assessment only, seals evaluation readiness, and stops. The same runtime may be rerun only when its existing checkout and local artifacts remain exact. It cannot launch 100k training, pilots, ablations, P:0007, or P:0009.

In [ ]:
from pathlib import Path
import os, re, subprocess

REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
TRAINING_EVIDENCE_COMMIT = '82633d66e5ea47f96b149ea22cc192fcf4526f06'
OPERATOR_IMPLEMENTATION_COMMIT = 'c26df4ca7541a6e44ca42343b326ffbf19fa9644'
if OPERATOR_IMPLEMENTATION_COMMIT == '__OPERATOR_IMPLEMENTATION_COMMIT__':
    raise RuntimeError('Notebook sealing commit has not pinned the operator implementation.')
if re.fullmatch(r'[0-9a-f]{40}', OPERATOR_IMPLEMENTATION_COMMIT) is None:
    raise ValueError('Operator implementation must be an exact 40-character Git SHA.')
REPO_DIR = Path('/content/MRIxFields-stage2-gate01-recovery-v8-' + OPERATOR_IMPLEMENTATION_COMMIT[:12])
def git_probe(repo_dir, *args):
    read_only_env = os.environ.copy()
    read_only_env['GIT_OPTIONAL_LOCKS'] = '0'
    return subprocess.run(
        ['git', *args], cwd=repo_dir, text=True, capture_output=True,
        env=read_only_env,
    )
def validate_existing_operator_checkout(repo_dir, repository_url, operator_commit, training_commit):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise RuntimeError('Existing operator checkout path is not a directory.')
    def required_text(*args):
        result = git_probe(repo_dir, *args)
        if result.returncode:
            raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
        return result.stdout.strip()
    if required_text('rev-parse', '--is-inside-work-tree') != 'true':
        raise RuntimeError('Existing operator path is not a Git worktree.')
    if Path(required_text('rev-parse', '--show-toplevel')).resolve() != repo_dir.resolve():
        raise RuntimeError('Existing operator checkout is not the expected worktree root.')
    if required_text('remote').splitlines() != ['origin']:
        raise RuntimeError('Existing operator checkout must have exactly one origin remote.')
    origins = required_text('remote', 'get-url', '--all', 'origin').splitlines()
    if origins != [repository_url]:
        raise RuntimeError('Existing operator checkout origin differs from the pinned repository.')
    if required_text('rev-parse', 'HEAD') != operator_commit:
        raise RuntimeError('Existing operator checkout is at the wrong commit.')
    symbolic = git_probe(repo_dir, 'symbolic-ref', '-q', 'HEAD')
    if symbolic.returncode != 1 or symbolic.stdout.strip():
        raise RuntimeError('Existing operator checkout must have a detached HEAD.')
    if required_text('status', '--porcelain=v1', '--untracked-files=all'):
        raise RuntimeError('Existing operator checkout is dirty.')
    ancestor = git_probe(repo_dir, 'merge-base', '--is-ancestor', training_commit, 'HEAD')
    if ancestor.returncode != 0:
        raise RuntimeError('Operator commit does not descend from the training-evidence commit.')
    changed_paths = required_text('diff', '--name-only', training_commit, 'HEAD').splitlines()
    allowed_exact = {
        'src/fieldbridge/evaluation/stage2_unified_gate01_p0006.py',
        'src/fieldbridge/evaluation/stage2_unified_preflight.py',
    }
    allowed_prefixes = ('notebooks/', 'tests/', 'docs/')
    disallowed = [path for path in changed_paths if path not in allowed_exact and not path.startswith(allowed_prefixes)]
    if disallowed:
        raise RuntimeError({'operator_diff_touches_training_critical_code': disallowed})
    training_critical_prefixes = (
        'src/fieldbridge/models/', 'src/fieldbridge/training/', 'configs/',
        'src/fieldbridge/data/photometry_factored_latent_bank.py',
        'src/fieldbridge/data/photometry_factored_bank_dataset.py',
    )
    if any(path.startswith(training_critical_prefixes) for path in changed_paths):
        raise RuntimeError('Training, model, optimizer, bank, config, or loss code changed.')
    return {'changed_path_count': len(changed_paths), 'checkout_reused': True}
checkout_reused = REPO_DIR.exists()
if checkout_reused:
    checkout_state = validate_existing_operator_checkout(REPO_DIR, REPOSITORY_URL, OPERATOR_IMPLEMENTATION_COMMIT, TRAINING_EVIDENCE_COMMIT)
else:
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', 'fetch', 'origin', OPERATOR_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', OPERATOR_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    checkout_state = validate_existing_operator_checkout(REPO_DIR, REPOSITORY_URL, OPERATOR_IMPLEMENTATION_COMMIT, TRAINING_EVIDENCE_COMMIT)
    checkout_state['checkout_reused'] = False
def git_text(*args):
    result = git_probe(REPO_DIR, *args)
    if result.returncode:
        raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
    return result.stdout.strip()
CLI_ENV = os.environ.copy()
print({'training_evidence_commit': TRAINING_EVIDENCE_COMMIT, 'operator_implementation_commit': OPERATOR_IMPLEMENTATION_COMMIT, 'operator_descends_from_training_evidence': True, 'training_critical_code_byte_identical': True, 'detached_clean_checkout': True, 'checkout_reused_without_mutation': checkout_state['checkout_reused'], 'cpu_high_ram_recommended': True, 'gpu_used': False}, flush=True)
operator_path = REPO_DIR / 'notebooks/stage2_gate01_legacy_recovery_operator.py'
exec(compile(operator_path.read_text(encoding='utf-8'), str(operator_path), 'exec'), globals())
